# PRISM - CONCH + BRACS
# Pathology Reliability In Scarce-label Medicine

In [1]:
import torch
import numpy as np
import pandas as pd
import os
from google.colab import userdata, drive
from torchvision import transforms

drive.mount('/content/drive')

BASE_DIR = '/content/drive/MyDrive/PRISM'
EMBED_DIR = f'{BASE_DIR}/embeddings/conch/bracs'
RESULTS_DIR = f'{BASE_DIR}/results'
os.makedirs(EMBED_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Mounted at /content/drive
GPU: NVIDIA A100-SXM4-80GB
GPU Memory: 85.1 GB


In [2]:
!pip install transformers timm einops huggingface_hub scikit-learn scipy tqdm -q
!pip install git+https://github.com/Mahmoodlab/CONCH.git -q

from huggingface_hub import login
login(token=userdata.get('HF_TOKEN'))
print("Login OK")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.5 MB/s eta 0:00:00
Login OK


In [3]:
from conch.open_clip_custom import create_model_from_pretrained
model, transform = create_model_from_pretrained(
    'conch_ViT-B-16',
    checkpoint_path='hf_hub:MahmoodLab/CONCH',
    hf_auth_token=userdata.get('HF_TOKEN')
)
model = model.cuda().eval()
print("CONCH loaded!")
print(f"GPU memory: {torch.cuda.memory_allocated()/1e9:.2f} GB")

/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


meta.yaml:   0%|          | 0.00/37.0 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/802M [00:00<?, ?B/s]

CONCH loaded!
GPU memory: 1.58 GB


In [4]:
# CONCH provides its own transform above

In [5]:
from PIL import Image
Image.MAX_IMAGE_PIXELS = None

from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader

DATASET_DIR = '/content/drive/MyDrive/PRISM/datasets/bracs/latest_version'
train_dataset = ImageFolder(root=f'{DATASET_DIR}/train', transform=transform)
val_dataset   = ImageFolder(root=f'{DATASET_DIR}/val',   transform=transform)
test_dataset  = ImageFolder(root=f'{DATASET_DIR}/test',  transform=transform)
print(f"Classes: {train_dataset.classes}")

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=False, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=64, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=64, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train: {len(train_dataset):,}")
print(f"Val:   {len(val_dataset):,}")
print(f"Test:  {len(test_dataset):,}")

Classes: ['0_N', '1_PB', '2_UDH', '3_FEA', '4_ADH', '5_DCIS', '6_IC']
Train: 3,657
Val:   312
Test:  570


In [6]:
from tqdm import tqdm

def extract_features(model, loader, device='cuda'):
    all_features, all_labels = [], []
    model.eval()
    with torch.no_grad():
        for images, labels in tqdm(loader, desc="Extracting"):
            images = images.to(device)
            features = model.encode_image(images, proj_contrast=False, normalize=True)
            features = features / features.norm(dim=-1, keepdim=True)
            all_features.append(features.cpu().numpy())
            all_labels.append(labels.numpy() if hasattr(labels, 'numpy') else np.array(labels))
    return np.concatenate(all_features), np.concatenate(all_labels)

print("Extracting train features...")
train_features, train_labels = extract_features(model, train_loader)
print(f"Train: {train_features.shape}")

print("Extracting val features...")
val_features, val_labels = extract_features(model, val_loader)
print(f"Val: {val_features.shape}")

print("Extracting test features...")
test_features, test_labels = extract_features(model, test_loader)
print(f"Test: {test_features.shape}")

Extracting train features...


Extracting: 100%|██████████| 58/58 [27:49<00:00, 28.79s/it]


Train: (3657, 512)
Extracting val features...


Extracting: 100%|██████████| 5/5 [04:12<00:00, 50.51s/it]


Val: (312, 512)
Extracting test features...


Extracting: 100%|██████████| 9/9 [04:53<00:00, 32.58s/it]

Test: (570, 512)


In [7]:
np.save(f'{EMBED_DIR}/train_features.npy', train_features)
np.save(f'{EMBED_DIR}/train_labels.npy',   train_labels)
np.save(f'{EMBED_DIR}/val_features.npy',   val_features)
np.save(f'{EMBED_DIR}/val_labels.npy',     val_labels)
np.save(f'{EMBED_DIR}/test_features.npy',  test_features)
np.save(f'{EMBED_DIR}/test_labels.npy',    test_labels)
print(f"Embeddings saved to: {EMBED_DIR}")

Embeddings saved to: /content/drive/MyDrive/PRISM/embeddings/conch/bracs


In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, f1_score, brier_score_loss

def compute_ece_multiclass(y_true, y_prob, n_bins=15):
    confidence  = y_prob.max(axis=1)
    predictions = y_prob.argmax(axis=1)
    correct     = (predictions == y_true).astype(float)
    bins = np.linspace(0, 1, n_bins + 1)
    ece  = 0.0
    for i in range(n_bins):
        mask = (confidence >= bins[i]) & (confidence < bins[i+1])
        if mask.sum() > 0:
            ece += mask.sum() * abs(correct[mask].mean() - confidence[mask].mean())
    return ece / len(y_true)

def run_label_fraction(train_f, train_l, val_f, val_l, test_f, test_l, fraction, seed=42):
    np.random.seed(seed)
    n   = int(len(train_l) * fraction)
    idx = np.random.choice(len(train_l), n, replace=False)
    clf = LogisticRegression(max_iter=1000, C=1.0, random_state=seed)
    clf.fit(train_f[idx], train_l[idx])
    y_prob    = clf.predict_proba(test_f)
    y_pred    = clf.predict(test_f)
    n_classes = y_prob.shape[1]
    return {
        'model': 'CONCH', 'dataset': 'BRACS',
        'fraction': fraction, 'n_samples': n, 'seed': seed,
        'auroc': roc_auc_score(test_l, y_prob, multi_class='ovr', average='macro'),
        'f1':    f1_score(test_l, y_pred, average='macro'),
        'ece':   compute_ece_multiclass(test_l, y_prob),
        'brier': np.mean([brier_score_loss((test_l==c).astype(int), y_prob[:,c]) for c in range(n_classes)]),
    }

fractions = [0.01, 0.05, 0.10, 0.25, 0.50, 1.00]
seeds     = [42, 123, 456]
results   = []

for frac in fractions:
    for seed in seeds:
        res = run_label_fraction(
            train_features, train_labels,
            val_features, val_labels,
            test_features, test_labels,
            fraction=frac, seed=seed
        )
        results.append(res)
        print(f"  {frac*100:.0f}% | seed {seed} | AUROC: {res['auroc']:.4f} | ECE: {res['ece']:.4f} | Brier: {res['brier']:.4f}")

df = pd.DataFrame(results)
print("\n--- Summary (mean over seeds) ---")
print(df.groupby('fraction')[['auroc','ece','brier','f1']].mean().round(4))

  1% | seed 42 | AUROC: 0.8219 | ECE: 0.1162 | Brier: 0.1084
  1% | seed 123 | AUROC: 0.8351 | ECE: 0.2225 | Brier: 0.1054
  1% | seed 456 | AUROC: 0.8424 | ECE: 0.0428 | Brier: 0.1125
  5% | seed 42 | AUROC: 0.8865 | ECE: 0.1704 | Brier: 0.0862
  5% | seed 123 | AUROC: 0.8790 | ECE: 0.0633 | Brier: 0.0904
  5% | seed 456 | AUROC: 0.8845 | ECE: 0.1828 | Brier: 0.0845
  10% | seed 42 | AUROC: 0.8911 | ECE: 0.1084 | Brier: 0.0810
  10% | seed 123 | AUROC: 0.8881 | ECE: 0.0658 | Brier: 0.0832
  10% | seed 456 | AUROC: 0.8912 | ECE: 0.1383 | Brier: 0.0792
  25% | seed 42 | AUROC: 0.8967 | ECE: 0.0889 | Brier: 0.0748
  25% | seed 123 | AUROC: 0.9020 | ECE: 0.0474 | Brier: 0.0771
  25% | seed 456 | AUROC: 0.8962 | ECE: 0.0692 | Brier: 0.0750
  50% | seed 42 | AUROC: 0.9022 | ECE: 0.0523 | Brier: 0.0731
  50% | seed 123 | AUROC: 0.9013 | ECE: 0.0416 | Brier: 0.0735
  50% | seed 456 | AUROC: 0.9015 | ECE: 0.0476 | Brier: 0.0733
  100% | seed 42 | AUROC: 0.9047 | ECE: 0.0298 | Brier: 0.0719
  1

In [9]:
from scipy.optimize import minimize_scalar

def find_temperature_multiclass(val_logits, val_labels):
    def nll(T):
        scaled = val_logits / T
        exp_s  = np.exp(scaled - scaled.max(axis=1, keepdims=True))
        probs  = exp_s / exp_s.sum(axis=1, keepdims=True)
        return -np.log(probs[np.arange(len(val_labels)), val_labels] + 1e-10).mean()
    return minimize_scalar(nll, bounds=(0.1, 10.0), method='bounded').x

def run_temperature_scaling(train_f, train_l, val_f, val_l, test_f, test_l, fraction, seed=42):
    np.random.seed(seed)
    n   = int(len(train_l) * fraction)
    idx = np.random.choice(len(train_l), n, replace=False)
    clf = LogisticRegression(max_iter=1000, C=1.0, random_state=seed)
    clf.fit(train_f[idx], train_l[idx])
    val_logits    = clf.decision_function(val_f)
    T             = find_temperature_multiclass(val_logits, val_l)
    test_prob_raw = clf.predict_proba(test_f)
    ece_raw       = compute_ece_multiclass(test_l, test_prob_raw)
    test_logits   = clf.decision_function(test_f)
    scaled        = test_logits / T
    exp_s         = np.exp(scaled - scaled.max(axis=1, keepdims=True))
    test_prob_s   = exp_s / exp_s.sum(axis=1, keepdims=True)
    ece_scaled    = compute_ece_multiclass(test_l, test_prob_s)
    n_classes     = test_prob_raw.shape[1]
    return {
        'model': 'CONCH', 'dataset': 'BRACS',
        'fraction': fraction, 'seed': seed,
        'temperature': T,
        'ece_raw': ece_raw, 'ece_scaled': ece_scaled,
        'ece_improvement': ece_raw - ece_scaled,
        'auroc':       roc_auc_score(test_l, test_prob_raw, multi_class='ovr', average='macro'),
        'brier_raw':   np.mean([brier_score_loss((test_l==c).astype(int), test_prob_raw[:,c]) for c in range(n_classes)]),
        'brier_scaled':np.mean([brier_score_loss((test_l==c).astype(int), test_prob_s[:,c])   for c in range(n_classes)]),
    }

ts_results = []
for frac in fractions:
    for seed in seeds:
        res = run_temperature_scaling(
            train_features, train_labels,
            val_features, val_labels,
            test_features, test_labels,
            fraction=frac, seed=seed
        )
        ts_results.append(res)

df_ts = pd.DataFrame(ts_results)
print("--- Temperature Scaling Results ---")
print(df_ts.groupby('fraction')[['temperature','ece_raw','ece_scaled','ece_improvement']].mean().round(4))

--- Temperature Scaling Results ---
          temperature  ece_raw  ece_scaled  ece_improvement
fraction                                                   
0.01           0.8149   0.1272      0.1180           0.0092
0.05           0.5996   0.1388      0.0637           0.0752
0.10           0.7106   0.1042      0.0699           0.0343
0.25           0.7791   0.0685      0.0519           0.0166
0.50           0.8784   0.0472      0.0406           0.0066
1.00           0.9660   0.0298      0.0247           0.0051


In [10]:
df.to_csv(f'{RESULTS_DIR}/conch_bracs_results.csv', index=False)
df_ts.to_csv(f'{RESULTS_DIR}/conch_bracs_temperature_scaling.csv', index=False)
print("Results saved!")
print(f"  {RESULTS_DIR}/conch_bracs_results.csv")
print(f"  {RESULTS_DIR}/conch_bracs_temperature_scaling.csv")

Results saved!
  /content/drive/MyDrive/PRISM/results/conch_bracs_results.csv
  /content/drive/MyDrive/PRISM/results/conch_bracs_temperature_scaling.csv
